# CREAM Graph Sanity Checks

Compares original, all-ones, identity, and random-same-edge-count graphs.


In [ ]:
from pathlib import Path
import pandas as pd
dataset_roots = {
  "celeba": "../experiments/CelebA/train_cbm/Standard_CelebA/celeba_graph_sanity/CREAM_celeba_graph_sanity",
  "cub": "../experiments/CUB/train_cbm/Standard_CUB/cub_graph_sanity/CREAM_cub_graph_sanity",
  "cfmnist": "../experiments/Complete_Concept_FMNIST/train_cbm/Standard_FashionMNIST/cfmnist_graph_sanity/CREAM_cfmnist_graph_sanity"
}
rows = []
for dataset, root_str in dataset_roots.items():
    root = Path(root_str)
    for csv_path in sorted(root.glob('*/last_metrics/*.csv')):
        df = pd.read_csv(csv_path)
        if df.empty:
            continue
        row = df.iloc[0].to_dict()
        row['dataset'] = dataset
        row['variant'] = csv_path.parents[1].name
        row['csv_path'] = str(csv_path)
        rows.append(row)
results = pd.DataFrame(rows)
cols = [c for c in ['dataset', 'variant', 'test_task_accuracy', 'test_concept_accuracy', 'test_dropout_task_accuracy', 'PFI_concept_importance', 'PFI_side_importance', 'num_trainable_parameters'] if c in results.columns]
results[cols].sort_values(['dataset', 'variant']) if not results.empty else results


In [ ]:
import matplotlib.pyplot as plt
if not results.empty and 'test_task_accuracy' in results.columns:
    for dataset, df in results.groupby('dataset'):
        ax = df.sort_values('variant').plot.bar(x='variant', y='test_task_accuracy', legend=False, figsize=(7, 4))
        ax.set_ylabel('Test task accuracy')
        ax.set_xlabel('Graph variant')
        ax.set_title(f'{dataset} CREAM Graph Sanity Check')
        plt.tight_layout()
        plt.show()


In [ ]:
import csv
dag_roots = {
  "celeba": "data/CelebA/graph_sanity",
  "cub": "data/CUB/graph_sanity",
  "cfmnist": "data/FashionMNIST/graph_sanity"
}
edge_counts = []
for dataset, dag_root in dag_roots.items():
    for path in sorted(Path('..', dag_root).glob('*.csv')):
        with open(path, newline='') as f:
            rows = list(csv.reader(f))
        edge_counts.append({
            'dataset': dataset,
            'dag': path.name,
            'true_entries': sum(cell == 'True' for row in rows[1:] for cell in row[1:]),
        })
pd.DataFrame(edge_counts)
